## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import subprocess

CISO_DIR  = '/content/CISO-SDM'
DATA_DIR  = '/content/drive/MyDrive/CISO/data'
CKPT_DIR  = '/content/drive/MyDrive/CISO/model_checkpoints_50_epochs_1339'
RESULTS_DIR = '/content/drive/MyDrive/CISO/results_CISO_plant_50_epochs_1339'

if not os.path.exists(CISO_DIR):
    subprocess.run(['git', 'clone', 'https://github.com/RolnickLab/CISO-SDM', CISO_DIR], check=True)
else:
    print("Repo already cloned, skipping.")

os.chdir(CISO_DIR)
print("Working directory:", os.getcwd())

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
if not os.path.exists(f'{CISO_DIR}/data'):
    os.symlink(DATA_DIR, f'{CISO_DIR}/data')
    print("Symlinked data/ → Drive")
else:
    print("Symlink already exists, skipping.")

!pip install "numpy<2.0" -q
!pip install -r requirements.txt -q

print("\nSetup complete ✓")
print(f"  repo:         {CISO_DIR}")
print(f"  data (Drive): {DATA_DIR}")
print(f"  ckpts (Drive): {CKPT_DIR}")

In [ ]:
import numpy as np
import pandas as pd
import json
import yaml
import shutil
from pathlib import Path

print("Imports OK")

## 1. Load Data

In [ ]:
DATA_CSV    = f'{DATA_DIR}/splotopen_global.csv'
SPLITS_JSON = f'{DATA_DIR}/splotopen_global_splits.json'

plants = pd.read_csv(DATA_CSV)
plants = plants.reset_index(drop=True)

with open(SPLITS_JSON) as f:
    plants_splits = json.load(f)

print(f"Loaded {len(plants):,} rows × {len(plants.columns):,} columns")
print(f"Split keys: {list(plants_splits.keys())}")
plants.head(2)

In [ ]:
env_cols     = [c for c in plants.columns if c.startswith('env_')]
species_cols = [c for c in plants.columns
                if c not in env_cols + ['time', 'latitude', 'longitude']]

worldclim_cols = [c for c in env_cols if 'bio' in c.lower()]
soilgrid_cols  = [c for c in env_cols if 'bio' not in c.lower()]

print(f"WorldClim cols ({len(worldclim_cols)}): {worldclim_cols}")
print(f"SoilGrid cols  ({len(soilgrid_cols)}):  {soilgrid_cols}")
print(f"Species cols   ({len(species_cols)}):   {species_cols[:5]} ...")

## 2. Build Split Indices

In [ ]:
print("First 5 train values:", plants_splits['train'][:5])
print("Type:", type(plants_splits['train'][0]))
print(f"train={len(plants_splits['train']):,}  "
      f"val={len(plants_splits['val']):,}  "
      f"test={len(plants_splits['test']):,}")
print(f"Total: {len(plants_splits['train'])+len(plants_splits['val'])+len(plants_splits['test']):,}  "
      f"vs rows: {len(plants):,}")

In [ ]:
train_split = np.array(plants_splits['train'])
val_split   = np.array(plants_splits['val'])
test_split  = np.array(plants_splits['test'])

print(f"train_split: {train_split.shape}, max={train_split.max()}")
print(f"val_split:   {val_split.shape},   max={val_split.max()}")
print(f"test_split:  {test_split.shape},  max={test_split.max()}")
assert train_split.max() < len(plants), "Train index out of bounds!"
assert val_split.max()   < len(plants), "Val index out of bounds!"
assert test_split.max()  < len(plants), "Test index out of bounds!"
print("✓ All indices in bounds")

## 3. Prepare Files in CISO's Expected Format

In [ ]:
# WorldClim: env_bio01 → bio_1, env_bio02 → bio_2, etc.
worldclim_rename = {}
for c in worldclim_cols:
    # handles env_bio01, env_bio1, env_BIO01, etc.
    num = ''.join(filter(str.isdigit, c)).lstrip('0') or '0'
    worldclim_rename[c] = f'bio_{num}'

print("WorldClim rename map:")
for k, v in worldclim_rename.items():
    print(f"  {k} → {v}")

In [ ]:
# SoilGrid column mapping
print("Your soilgrid cols:", soilgrid_cols)

soilgrid_rename = {
    # 'env_ORCDRC': 'ORCDRC',   # ← uncomment and edit as needed
    # 'env_PHIHOX': 'PHIHOX',
    # 'env_CECSOL': 'CECSOL',
    # 'env_BDTICM': 'BDTICM',
    # 'env_CLYPPT': 'CLYPPT',
    # 'env_SLTPPT': 'SLTPPT',
    # 'env_SNDPPT': 'SNDPPT',
    # 'env_BLDFIE': 'BLDFIE',
}

# Default mapping for env_-prefixed columns.
if not soilgrid_rename:
    soilgrid_rename = {c: c.replace('env_', '').upper() for c in soilgrid_cols}
    print("Auto-built soilgrid rename map:")
    for k, v in soilgrid_rename.items():
        print(f"  {k} → {v}")

In [ ]:
OUT_DIR = Path("data/sPlotOpen")
OUT_DIR.mkdir(parents=True, exist_ok=True)

plants['PlotObservationID'] = plants.index

plants[['PlotObservationID'] + worldclim_cols]\
    .rename(columns=worldclim_rename)\
    .to_csv(OUT_DIR / "worldclim_data.csv", index=False)

plants[['PlotObservationID'] + soilgrid_cols]\
    .rename(columns=soilgrid_rename)\
    .to_csv(OUT_DIR / "soilgrid_data.csv", index=False)

# Filter species: >= 100 occurrences
targets_full         = plants[species_cols].to_numpy().astype(np.float32)
species_counts       = targets_full.sum(axis=0)
keep                 = species_counts >= 100
targets              = targets_full[:, keep]
species_cols_filtered = [s for s, k in zip(species_cols, keep) if k]

print(f"Species before filtering: {len(species_cols):,}")
print(f"Species after  filtering: {len(species_cols_filtered):,}")

pd.DataFrame({'species': species_cols_filtered})\
    .to_csv(OUT_DIR / "species_merge_duplicates_v2.csv", index=False)

np.save(OUT_DIR / "merged_species_occurrences_v2.npy", targets)

np.save(OUT_DIR / "train_indices.npy",      train_split)
np.save(OUT_DIR / "validation_indices.npy", val_split)
np.save(OUT_DIR / "test_indices.npy",       test_split)

print("\nFiles saved:")
for f in sorted(OUT_DIR.iterdir()):
    print(f"  {f.name}")

## 4. Verify Data Integrity

In [ ]:
wc  = pd.read_csv(OUT_DIR / "worldclim_data.csv")
sg  = pd.read_csv(OUT_DIR / "soilgrid_data.csv")
sp  = pd.read_csv(OUT_DIR / "species_merge_duplicates_v2.csv")
tgt = np.load(OUT_DIR / "merged_species_occurrences_v2.npy")
tr  = np.load(OUT_DIR / "train_indices.npy")
val = np.load(OUT_DIR / "validation_indices.npy")
te  = np.load(OUT_DIR / "test_indices.npy")

print(f"worldclim  : {wc.shape}   cols: {wc.columns.tolist()}")
print(f"soilgrid   : {sg.shape}   cols: {sg.columns.tolist()}")
print(f"species    : {sp.shape}   cols: {sp.columns.tolist()}")
print(f"targets    : {tgt.shape}")
print(f"train/val/test: {tr.shape} / {val.shape} / {te.shape}")

n_env = len(wc.columns) - 1 + len(sg.columns) - 1
assert tgt.shape[1] == len(sp), \
    f"Species mismatch: targets has {tgt.shape[1]} but species list has {len(sp)}"
assert len(wc) == len(tgt), \
    f"Row mismatch: worldclim {len(wc)} vs targets {len(tgt)}"
assert tr.max() < len(tgt) and val.max() < len(tgt) and te.max() < len(tgt), \
    "Split index out of bounds!"

print(f"\n✓ All checks passed")
print(f"  input_dim  (for config) = {n_env}")
print(f"  num_classes (for config) = {tgt.shape[1]}")

## 5. Write Config File

In [ ]:
env_columns_for_config = (
    [worldclim_rename[c] for c in worldclim_cols] +
    [soilgrid_rename[c]  for c in soilgrid_cols]
)
print(f"env_columns ({len(env_columns_for_config)}):")
print(env_columns_for_config)

In [ ]:
CONFIG_PATH = Path("configs/config_ciso_mydata.yaml")
CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)

# Edit these
EXPERIMENT_NAME = "splot_ciso_mydata"
CHECKPOINT_DIR  = CKPT_DIR
MAX_EPOCHS      = 50
BATCH_SIZE      = 64
LEARNING_RATE   = 1e-3

config_dict = {
    'mode': 'train',
    'dataset_name': 'sPlot',
    'model': {
        'name': 'CISOModel',
        'input_dim': n_env,
        'hidden_dim': 256,
        'num_classes': int(tgt.shape[1]),
        'backbone': 'SimpleMLPBackbone',
    },
    'training': {
        'seed': 1339,
        'learning_rate': LEARNING_RATE,
        'max_epochs': MAX_EPOCHS,
        'accelerator': 'gpu',
        'devices': 1,
    },
    'logger': {
        'project_name': 'sPlotOpen',
        'experiment_name': EXPERIMENT_NAME,
        'experiment_key': '',
        'checkpoint_path': CHECKPOINT_DIR,
        'checkpoint_name': '',
        'save_preds_path': '',
    },
    'data': {
        'dataloader_to_use': 'sPlotMaskedDataset',
        'base': 'data/sPlotOpen',
        'train': 'train_indices.npy',
        'validation': 'validation_indices.npy',
        'test': 'test_indices.npy',
        'targets': 'merged_species_occurrences_v2.npy',
        'worldclim_data_path': 'worldclim_data.csv',
        'soilgrid_data_path': 'soilgrid_data.csv',
        'species_list': 'species_merge_duplicates_v2.csv',
        'species_occurrences_threshold': 100,
        'batch_size': BATCH_SIZE,
        'env_columns': env_columns_for_config,
        'partial_labels': {
            'use': True,
            'quantized_mask_bins': 1,      # binary presence/absence
            'train_known_ratio': 0.75,     # max 75% species known during training
            'eval_known_ratio': 0,         # fully unconditioned at eval
            'predict_family_of_species': -1,  # -1 = predict all species
        },
    },
}

with open(CONFIG_PATH, 'w') as f:
    yaml.dump(config_dict, f, default_flow_style=False, sort_keys=False)

print(f"Config written to: {CONFIG_PATH}")
print()
print(open(CONFIG_PATH).read())

## 6. Train

In [ ]:
import os
os.environ['COMET_MODE'] = 'OFFLINE'
os.environ.setdefault('COMET_OFFLINE_DIRECTORY', 'comet_logs')
os.makedirs('comet_logs', exist_ok=True)

In [ ]:
!python main.py --help

In [ ]:
!python main.py --config configs/config_ciso_mydata.yaml

In [ ]:
ckpt_dir = Path(CHECKPOINT_DIR)
if ckpt_dir.exists():
    ckpts = list(ckpt_dir.glob("**/*.ckpt")) + list(ckpt_dir.glob("**/*.pt"))
    print("Saved checkpoints:")
    for c in ckpts:
        print(f"  {c}")
else:
    print(f"Checkpoint dir not found: {ckpt_dir}")

## 7. Benchmark at Multiple Masking Levels

Evaluate CISO at `eval_known_ratio` ∈ {0.0, 0.25, 0.5, 0.75, 1.0}, corresponding to masking fractions p ∈ {1.0, 0.75, 0.5, 0.25, 0.0} — matching the four masking levels reported in STEM-LM.

In [ ]:
# Random-mask sweep matching STEM-LM's --val_p_list. p=0 (eval_known_ratio=1.0)
# is omitted: all species revealed = trivial copy-through, not a real prediction.
EVAL_RATIOS = [0.0, 0.25, 0.5, 0.75]

CHECKPOINT_NAME = ""
ckpts = list(Path(CKPT_DIR).glob("**/*.ckpt"))
best = [p for p in ckpts if 'last' not in p.name]
if best:
    CHECKPOINT_NAME = str(sorted(best)[-1])
elif ckpts:
    CHECKPOINT_NAME = str(sorted(ckpts)[-1])

if CHECKPOINT_NAME:
    print(f"Auto-detected checkpoint: {CHECKPOINT_NAME}")
else:
    print("No checkpoint found — run training first")


In [ ]:
Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)

for ratio in EVAL_RATIOS:
    print(f"\n{'='*60}")
    print(f"Evaluating eval_known_ratio={ratio}  (mask p={1-ratio:.2f})")
    print('='*60, flush=True)

    ratio_str = str(ratio).replace('.', 'p')

    preds_dir = f'{RESULTS_DIR}/preds_ratio_{ratio_str}'
    Path(preds_dir).mkdir(parents=True, exist_ok=True)

    test_config = config_dict.copy()
    test_config['mode'] = 'test'
    test_config['logger'] = config_dict['logger'].copy()
    test_config['logger']['checkpoint_name'] = CHECKPOINT_NAME
    test_config['logger']['save_preds_path'] = preds_dir
    test_config['data'] = config_dict['data'].copy()
    test_config['data']['partial_labels'] = config_dict['data']['partial_labels'].copy()
    test_config['data']['partial_labels']['eval_known_ratio'] = ratio

    cfg_path = f"configs/config_ciso_test_{ratio_str}.yaml"
    res_csv  = f"{RESULTS_DIR}/results_ratio_{ratio_str}.csv"

    with open(cfg_path, 'w') as f:
        yaml.dump(test_config, f, default_flow_style=False, sort_keys=False)

    !python main.py --config {cfg_path} --results_file_name {res_csv}

print("\nDone — all masking levels evaluated.")

## 8. Summarise Results

In [ ]:
import os, yaml, json
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from sklearn.metrics import roc_auc_score, average_precision_score
from scipy.stats import spearmanr

from src.config import Config
from src.dataloaders.splot_dataloader import sPlotDataModule
from src.trainers.splot_trainer import sPlotTrainer

def _safe_auc_roc(y, p):
    if y.size == 0 or len(set(y.tolist())) < 2 or np.isnan(p).any(): return float('nan')
    try: return float(roc_auc_score(y, p))
    except Exception: return float('nan')

def _safe_auc_pr(y, p):
    if y.size == 0 or y.sum() == 0 or y.sum() == y.size or np.isnan(p).any(): return float('nan')
    try: return float(average_precision_score(y, p))
    except Exception: return float('nan')

def _safe_brier(y, p):
    if y.size == 0 or np.isnan(p).any(): return float('nan')
    return float(np.mean((p - y.astype(np.float64))**2))

def _safe_ece(y, p, n_bins=15):
    if y.size == 0 or np.isnan(p).any(): return float('nan')
    edges = np.linspace(0, 1, n_bins+1)
    idx = np.clip(np.digitize(p, edges) - 1, 0, n_bins-1)
    err = 0.0; n = p.size
    for b in range(n_bins):
        m = idx == b
        if not m.any(): continue
        err += (m.sum()/n) * abs(y[m].mean() - p[m].mean())
    return float(err)

def _safe_cbi(y, p, n_windows=101, width=0.1, min_per_window=10):
    if y.size == 0 or y.sum() == 0 or y.sum() == y.size or np.isnan(p).any(): return float('nan')
    pres = p[y==1]; bg = p[y==0]
    if pres.size == 0 or bg.size == 0: return float('nan')
    centers = np.linspace(0, 1, n_windows); half = width/2
    pe = np.full(n_windows, np.nan)
    for i, c in enumerate(centers):
        lo, hi = c-half, c+half
        n_bg = int(((bg>=lo)&(bg<=hi)).sum())
        if n_bg < min_per_window: continue
        e = n_bg/bg.size
        if e == 0: continue
        pe[i] = ((pres>=lo)&(pres<=hi)).sum()/pres.size / e
    ok = np.isfinite(pe)
    if ok.sum() < 3 or np.unique(pe[ok]).size < 2: return float('nan')
    try:
        rho = spearmanr(centers[ok], pe[ok]).statistic
        return float(rho) if np.isfinite(rho) else float('nan')
    except Exception: return float('nan')


def _per_species_metrics(probs, targets, masks):
    """Per-species metrics, FILTERED to rows where each species was masked.
    probs, targets, masks: (N, S) float / int / int. mask == -1 => masked target."""
    S = probs.shape[1]
    out = {k: {} for k in ['auc_roc','auc_pr','cbi','brier','ece']}
    n_kept_per_sp = []
    for s in range(S):
        keep = masks[:, s] == -1
        n_kept_per_sp.append(int(keep.sum()))
        if keep.sum() == 0:
            continue
        y = targets[keep, s].astype(np.int64)
        p = probs[keep, s].astype(np.float64)
        if y.sum() == 0 or y.sum() == y.size:
            continue
        out['auc_roc'][s] = _safe_auc_roc(y, p)
        out['auc_pr'][s]  = _safe_auc_pr(y, p)
        out['cbi'][s]     = _safe_cbi(y, p)
        out['brier'][s]   = _safe_brier(y, p)
        out['ece'][s]     = _safe_ece(y, p)
    return out, n_kept_per_sp

def _summarize(per_sp):
    def cl(d): return [v for v in d.values() if np.isfinite(v)]
    aucs = cl(per_sp['auc_roc']); prs = cl(per_sp['auc_pr'])
    cbis = cl(per_sp['cbi']);     bri = cl(per_sp['brier']);  ece = cl(per_sp['ece'])
    q = lambda x, p: float(np.quantile(x, p)) if x else float('nan')
    return {
        'mean_auc_roc': float(np.mean(aucs)) if aucs else float('nan'),
        'auc_roc_q25': q(aucs, .25), 'auc_roc_q50': q(aucs, .50), 'auc_roc_q75': q(aucs, .75),
        'mean_auc_pr': float(np.mean(prs)) if prs else float('nan'),
        'mean_cbi':    float(np.mean(cbis)) if cbis else float('nan'),
        'mean_brier':  float(np.mean(bri)) if bri else float('nan'),
        'mean_ece':    float(np.mean(ece)) if ece else float('nan'),
        'n_species':   len(aucs),
    }


def _get_seed(run_id, seed): return (run_id * (seed + (run_id - 1))) % (2**31 - 1)

def _run_inference_with_masks(test_cfg_path):
    """Load CISO checkpoint + run on test loader; return (probs, targets, masks) numpy.
    Captures the per-row mask tensor produced by sPlotMaskedDataset so that
    metrics can be filtered to only-masked species (matching STEM-LM).
    GPU-enabled: model and batches moved to CUDA when available."""
    cfg  = Config(**yaml.safe_load(open(test_cfg_path)))
    seed = _get_seed(1, cfg.training.seed)
    ckpt = os.path.join(cfg.logger.checkpoint_path, cfg.logger.experiment_name,
                        str(seed), cfg.logger.checkpoint_name)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    dm   = sPlotDataModule(cfg.data); dm.setup()
    task = sPlotTrainer(cfg).to(device)
    task.load_state_dict(torch.load(ckpt, map_location=device)['state_dict'])
    task.eval()

    probs_all, tgt_all, mask_all = [], [], []
    with torch.no_grad():
        for batch in dm.test_dataloader(num_workers=0, persistent_workers=False):
            batch_dev = {k: (v.to(device) if torch.is_tensor(v) else v)
                         for k, v in batch.items()}
            logits = task(batch_dev)
            probs_all.append(torch.sigmoid(logits).cpu().numpy())
            tgt_all.append(batch_dev['targets'].cpu().numpy())
            mask_all.append(batch_dev['mask'].cpu().numpy())  # -1 = masked, 0/1 = known
    return (np.concatenate(probs_all, 0),
            np.concatenate(tgt_all, 0).astype(np.int64),
            np.concatenate(mask_all, 0).astype(np.int64))


# Full STEM-LM metric set per eval_known_ratio
rows = []
for ratio in EVAL_RATIOS:
    rs = str(ratio).replace('.', 'p')
    cfg_path = f"configs/config_ciso_test_{rs}.yaml"
    if not Path(cfg_path).exists():
        print(f"missing {cfg_path} — skipping"); continue
    print(f"\n→ ratio={ratio} (mask p={1-ratio:.2f})", flush=True)
    probs, tgt, mask = _run_inference_with_masks(cfg_path)
    per_sp, n_kept = _per_species_metrics(probs, tgt, mask)
    summ = _summarize(per_sp)
    summ.update({'eval_known_ratio': ratio, 'masking_p': round(1-ratio, 2),
                 'avg_masked_rows_per_sp': float(np.mean(n_kept))})
    rows.append(summ)
    print({k: round(v, 4) if isinstance(v, float) else v for k, v in summ.items()})

summary = pd.DataFrame(rows).set_index('masking_p').sort_index()
cols = ['mean_auc_roc','auc_roc_q25','auc_roc_q50','auc_roc_q75',
        'mean_auc_pr','mean_cbi','mean_brier','mean_ece','n_species','avg_masked_rows_per_sp']
summary = summary[['eval_known_ratio'] + cols]
print("\n── CISO benchmark (STEM-LM-faithful, masked-only filter) ─────────")
print(summary.to_string())
out_csv = f"{RESULTS_DIR}/ciso_benchmark_summary.csv"
summary.to_csv(out_csv)
print(f"\nSaved to {out_csv}")